# Convert Input vectors of size 3x32x32 to 64 tokens

In [ ]:
import torch
import zarr
import numpy as np
import sys
import os
import re
from pathlib import Path
from zarr.storage import ZipStore
from torch.utils.data import Dataset
from omegaconf import OmegaConf
import bisect
import pytorch_lightning as pl
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from pytorch_lightning.callbacks import ModelCheckpoint
from omegaconf import OmegaConf
from zarr.storage import ZipStore
from torchvision.utils import save_image



class VQForOrders(pl.LightningModule):
    """ VQGAN model class, adapted for Orders """
    def __init__(self, vqmodel, lr=1e-4):
        super().__init__()
        self.m = vqmodel
        self.lr = lr

    def forward(self, x):
        q, _, info = self.m.encode(x)
        
        return self.m.decode(q)

    def training_step(self, batch, _):
        x = batch
        x_rec = self(x)
        loss = ((x_rec - x) ** 2).mean()
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x = batch
        x_rec = self(x)
        val_loss = ((x_rec - x) ** 2).mean()
        self.log("val_loss", val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)



def load_best_model(best_ckpt_path, repo_root):
    from ldm.util import instantiate_from_config

    cfg = OmegaConf.load(
        repo_root / "Mars_Derrick/third_party/latent-diffusion/models/first_stage_models/vq-f4/config.yaml"
    )

    vq = instantiate_from_config(cfg.model)

    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt_path,
        vqmodel=vq,
        lr=1e-4,
        strict=True
    )
    

    device = torch.device("cuda" if torch.cuda.is_available()
                          else "mps" if torch.backends.mps.is_available()
                          else "cpu")

    model.to(device)
    model.eval()

    return model, device

def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt files found in {ckpt_dir}")

    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        if m is None:
            raise ValueError(f"Could not parse val_loss from filename: {p.name}")
        return float(m.group(1))

    best_ckpt = min(ckpts, key=extract_val_loss)
    return best_ckpt
    

class FolderZarrDataset(Dataset):
    def __init__(self, path):


        self.arrays = []
        self.lengths = []

        
        zarr_name = path.name.replace(".zip", "")
        dataset_name = f"{zarr_name}/images"

        store = ZipStore(str(path), mode="r")
        arr = zarr.open(store=store, path=dataset_name, mode="r")

        print(f"Loaded {path} with shape {arr.shape}")

   

        self.arrays.append(arr)
        self.lengths.append(arr.shape[0])

        # cumulative lengths for indexing
        self.cum_lengths = []
        total = 0
        for l in self.lengths:
            total += l
            self.cum_lengths.append(total)

    def __len__(self):
        return self.cum_lengths[-1]

BATCH_SIZE = 2048  # adjust according to GPU memory
device = torch.device("cuda")

dataset_path = Path("/scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-TSLA-2025-12-22_order_images.zarr.zip")
OUT_DIR = Path("./latents_zip")
OUT_DIR.mkdir(exist_ok=True)

THIS_DIR = Path(os.getcwd()).resolve().parent
REPO = THIS_DIR.parents[1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sys.path.insert(0, str(REPO/ "Mars_Derrick" / "third_party" / "latent-diffusion"))
sys.path.insert(0, str(REPO/ "Mars_Derrick"  / "third_party" / "taming-transformers"))

#Finding best model 
BEST_CKPT_PATH = Path("/scratch") / "project_2012747" / "Mars_Derrick" / "checkpoints" / "checkpoint_downsample_100/" 
BEST_CKPT = find_best_checkpoint(BEST_CKPT_PATH)


# ---- Load model ----
model, device = load_best_model(BEST_CKPT, REPO)
Best_model = model

model = model.to(device)
model.eval()

dataset = FolderZarrDataset(dataset_path)

for file_idx, arr in enumerate(dataset.arrays):
    num_samples = arr.shape[0]
    original_file_name = dataset_path.stem
    out_file = OUT_DIR / f"{original_file_name}_tokens.zarr.zip"

    # Zarr store
    store = ZipStore(str(out_file), mode="w")
    tokens_arr = zarr.zeros(
        shape=(num_samples, 64),
        chunks=(BATCH_SIZE, 64),
        dtype="i4",
        store=store,
        overwrite=True
    )

    next_report = 50_000

    for start in range(0, num_samples, BATCH_SIZE):
        end = min(start + BATCH_SIZE, num_samples)
        batch_imgs = arr[start:end]

        # Convert to tensor once, normalize in one step
        x_input = torch.as_tensor(batch_imgs, dtype=torch.float32, device=device)
        x_input = x_input / 255.0  # simplest way


    

        with torch.inference_mode():
            _, _, info = model.m.encode(x_input)

            B = x_input.size(0)
            indices = info[2].view(B, 8, 8)
            tokens = indices.reshape(B, -1)  # flatten to 64

        # move to CPU once
        tokens_arr[start:end] = tokens.cpu().numpy()

        while end >= next_report:
            print(f"Processed {next_report}/{num_samples}")
            next_report += 50_000

    store.close()
    print(f"Saved token Zarr: {out_file} with shape {tokens_arr.shape}")


Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-GOOGL-2025-12-16_order_images.zarr.zip with shape (18751982, 3, 32, 32)
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.

Processing file 1/1 with 18751982 samples
original_file_name LOBSTER-GOOGL-2025-12-16_order_images.zarr


## Take a sample 64 vector and convert back to 3 dimensional (RGB) image of size 3x32x32

In [19]:
import torch
import zarr
from zarr.storage import ZipStore
from pathlib import Path

# --- path to your saved token Zarr file ---
zarr_file = OUT_DIR / "LOBSTER-GOOGL-2025-12-16_order_images.zarr_tokens.zarr.zip"

# --- load Zarr ---
store = ZipStore(str(zarr_file), mode="r")
tokens_arr = zarr.open(store, mode="r")

# --- pick a sample (e.g., first one) ---
tokens = torch.tensor(tokens_arr[0], dtype=torch.long)  # shape: (64,)

# --- reshape tokens back to grid (8x8) ---
tokens_grid = tokens.view(1, 8, 8)  # (1, 8, 8) batch dimension

# --- look up codebook embeddings ---
# model.m is your VQGAN
# codebook is usually nn.Embedding inside your VQGAN quantizer
codebook = model.m.quantize.embedding.weight  # shape: (8192, 3)
print(len(codebook))
# tokens → embeddings
z_q = codebook[tokens_grid]  # shape: (1, 8, 8, 3)

# --- permute to (B, C, H, W) for decoder ---
z_q = z_q.permute(0, 3, 1, 2).contiguous()  # (1, 3, 8, 8)

# --- decode to image ---
with torch.inference_mode():
    x_hat = model.m.decode(z_q)  # (1, 3, 32, 32), float tensor

# --- convert to numpy (optional) ---
x_hat_np = x_hat[0].cpu().numpy()
print(x_hat_np.shape)  # should be (3, 32, 32)


8192
(3, 32, 32)


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/users/edwardma/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/users/edwardma/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-TSLA-2025-12-22_order_images.zarr.zip with shape (5548388, 3, 32, 32)
Processed 50000/5548388
Processed 100000/5548388
Processed 150000/5548388
Processed 200000/5548388
Processed 250000/5548388
Processed 300000/5548388
Processed 350000/5548388
Processed 400000/5548388
Processed 450000/5548388
Processed 500000/5548388
Processed 550000/5548388
Processed 600000/5548388
Processed 650000/5548388
Processed 700000/5548388
Processed 750000/5548388
Processed 800000/5548388
Processed 850000/5548388
Processed 900000/5548388
Processed 950000/5548388
Processed 1000000/5548388
Processed 1050000/5548388
Processed 1100000/5548388
Processed 1150000/5548388
Processed 1200000/5548388
Processed 1250000/5548388
Processed 1300000/5548388
Processed 1350000/5548388
Processed 1400000/5548388
Processed 1450000/5548388
Proces